In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc
spark = SparkSession.builder.appName("sparksql").getOrCreate()


In [2]:
print(spark.version)


3.5.0


In [3]:
data = spark.read.format('csv').option("inferSchema","true").option("header","true").option("path","operations_management.csv").load()

In [4]:
data.printSchema()


root
 |-- description: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- level: integer (nullable = true)
 |-- size: string (nullable = true)
 |-- line_code: string (nullable = true)
 |-- value: integer (nullable = true)



In [5]:
data_2 = data.select("industry","value").filter((col("value") > 200) & (col("industry")!="total" )). orderBy(desc("value")) 

In [6]:
print(type(data_2))
data_2.show(10)

<class 'pyspark.sql.dataframe.DataFrame'>
+--------------------+-----+
|            industry|value|
+--------------------+-----+
|        Construction| 6030|
|        Construction| 5904|
|        Construction| 5229|
|Accommodation & f...| 5058|
|        Construction| 4965|
|        Construction| 4959|
|Accommodation & f...| 4950|
|        Construction| 4686|
|        Construction| 4668|
|        Construction| 4665|
+--------------------+-----+
only showing top 10 rows



In [7]:
data_2.createOrReplaceTempView("data")

In [8]:
spark.sql("SELECT industry,value FROM data where value> 200 and industry != 'total'").show(5)

+--------------------+-----+
|            industry|value|
+--------------------+-----+
|        Construction| 6030|
|        Construction| 5904|
|        Construction| 5229|
|Accommodation & f...| 5058|
|        Construction| 4965|
+--------------------+-----+
only showing top 5 rows



In [9]:
spark.catalog.listTables()

[Table(name='data', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [10]:
spark

In [14]:
data_temp = spark.read.format("csv").option("inferSchema","true").option("header","true").option("path","operations_management.csv").load()

In [20]:
data_temp_2 = data_temp.select(data_temp.industry,data_temp.level,data_temp.value).where(data_temp.value > 200 ).withColumn("plus1",data_temp.value+1)
                                                            

In [21]:
data_temp_2.show(5)

+--------------------+-----+-----+-----+
|            industry|level|value|plus1|
+--------------------+-----+-----+-----+
|               total|    0|13080|13081|
|               total|    0| 3348| 3349|
|               total|    0| 1089| 1090|
|               total|    0| 1023| 1024|
|Agriculture, fore...|    1| 2364| 2365|
+--------------------+-----+-----+-----+
only showing top 5 rows



In [22]:
data_temp_2.write.format("csv").mode("overwrite").option("path","result.csv").partitionBy("value").save()